In [1]:
import duckdb
import time

def extract_overture_india():
    print("Initializing DuckDB spatial engine...")
    con = duckdb.connect("overture_spatial.db")
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    # Base S3 Path for Overture (August 2026 Stable Release)
    S3_BASE = "s3://overturemaps-us-west-2/release/2026-08-19.0"
    
    # Bounding Box for India
    INDIA_BBOX = "bbox.xmin >= 68.1 AND bbox.xmax <= 97.4 AND bbox.ymin >= 6.5 AND bbox.ymax <= 37.1"

    # --- 1. PLACES (Businesses & Anchors) ---
    print("Streaming Places (Confidence >= 0.60)...")
    con.execute(f"""
        COPY (
            SELECT 
                id,
                names.primary AS name,
                categories.primary AS primary_category,
                confidence,
                phones[1] AS phone,
                websites[1] AS website,
                addresses[1].locality AS city,
                addresses[1].postcode AS postcode,
                geometry
            FROM read_parquet('{S3_BASE}/theme=places/type=place/*', filename=true, hive_partitioning=1)
            WHERE (
                addresses[1].country = 'IN' 
                OR ({INDIA_BBOX})
            )
              AND confidence >= 0.60
        ) TO 'india_places.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

extract_overture_india()

Initializing DuckDB spatial engine...
Streaming Places (Confidence >= 0.60)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))